# 08. Ensemble Modeling Strategies

**Objective:** Train and compare 8 base models + 4 ensemble methods for financial news classification

**Base Models (8):**
1. Logistic Regression
2. Random Forest
3. Gradient Boosting
4. XGBoost
5. LightGBM
6. Support Vector Machine (SVM)
7. Naive Bayes
8. K-Nearest Neighbors (KNN)

**Ensemble Methods (4):**
1. Hard Voting
2. Soft Voting (Weighted)
3. Bagging
4. Stacking (Meta-learner)

In [1]:
import pandas as pd
import numpy as np
import json
from pathlib import Path
from tqdm.auto import tqdm
import warnings
warnings.filterwarnings('ignore')

# Machine Learning
from sklearn.model_selection import train_test_split, TimeSeriesSplit, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, log_loss, confusion_matrix, classification_report
)

# Base Models
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

# Ensemble Methods
from sklearn.ensemble import VotingClassifier, BaggingClassifier, StackingClassifier

# Model Persistence
import joblib

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

print(" All libraries loaded successfully")

 All libraries loaded successfully


## 1. Setup Paths and Configuration

In [2]:
# Setup paths
BASE_DIR = Path.cwd().parent
DATA_DIR = BASE_DIR / '01_DATA'
FEATURES_DIR = DATA_DIR / 'features'
RESULTS_DIR = BASE_DIR / '03_RESULTS'
OUTPUTS_DIR = RESULTS_DIR / 'outputs'
VIZ_DIR = RESULTS_DIR / 'visualizations'
MODELS_DIR = BASE_DIR / '04_MODELS'
CONFIG_DIR = BASE_DIR / '06_CONFIG'

# Create model directories
BASE_MODELS_DIR = MODELS_DIR / 'base_models'
ENSEMBLE_DIR = MODELS_DIR / 'ensemble'
BASE_MODELS_DIR.mkdir(parents=True, exist_ok=True)
ENSEMBLE_DIR.mkdir(parents=True, exist_ok=True)

# Load configuration
with open(CONFIG_DIR / 'framework_config.json', 'r') as f:
    config = json.load(f)

with open(CONFIG_DIR / 'ensemble_config.json', 'r') as f:
    ensemble_config = json.load(f)

print(f" Base directory: {BASE_DIR}")
print(f" Models directory: {MODELS_DIR}")
print(f" Configuration loaded")

 Base directory: c:\Users\HARSHIT\Desktop\p\nlp analysis
 Models directory: c:\Users\HARSHIT\Desktop\p\nlp analysis\04_MODELS
 Configuration loaded


## 2. Load Feature Matrix

In [3]:
# Load final feature matrix from notebook 07
df = pd.read_csv(FEATURES_DIR / 'final_feature_matrix.csv')
df['date'] = pd.to_datetime(df['date'])

print(f" Loaded {len(df):,} articles")
print(f" Total features: {df.shape[1]}")
print(f" Date range: {df['date'].min()} to {df['date'].max()}")
print(f" Companies: {df['ticker'].nunique()}")

df.head()

 Loaded 63 articles
 Total features: 93
 Date range: 2025-09-26 07:00:00 to 2026-01-18 12:19:48
 Companies: 19


,date,title,text,source,url,ticker,publisher,query,article_length,word_count,...,topic_stability,hour,day_of_week,is_weekend,is_market_hours,days_since_last,recency_score,sentiment_lag_1,sentiment_lag_3,sentiment_lag_7
0,2026-01-14 08:02:00,This Unstoppable Stock Has 4 Catalysts to Fuel...,This Unstoppable Stock Has 4 Catalysts to Fuel...,Google News,https://news.google.com/rss/articles/CBMimAFBV...,AAPL,NaN,AAPL stock news,157,25,...,0.000000,8,2,0,0,0.000000,0.869966,NaN,NaN,NaN
1,2026-01-18 12:15:12,Apple stock price slips to $255 ahead of a hol...,Apple stock price slips to $255 ahead of a hol...,Google News,https://news.google.com/rss/articles/CBMiuAFBV...,AAPL,NaN,AAPL stock news,121,18,...,0.000000,12,6,1,1,4.175833,0.999894,-0.5719,NaN,NaN
2,2025-12-27 08:00:00,Is Weakness In Abbott Laboratories (NYSE:ABT) ...,Is Weakness In Abbott Laboratories (NYSE:ABT) ...,Google News,https://news.google.com/rss/articles/CBMiiwFBV...,ABT,NaN,ABT stock news,152,21,...,0.000000,8,5,1,0,0.000000,0.477425,NaN,NaN,NaN
3,2025-12-31 08:00:00,Here is What to Know Beyond Why Abbott Laborat...,Here is What to Know Beyond Why Abbott Laborat...,Google News,https://news.google.com/rss/articles/CBMiiAFBV...,ABT,NaN,ABT stock news,102,15,...,0.500000,8,2,0,0,4.000000,0.545521,-0.3818,NaN,NaN
4,2026-01-15 16:55:56,Dividend King Abbott Shows Why 52 Consecutive ...,Dividend King Abbott Shows Why 52 Consecutive ...,Google News,https://news.google.com/rss/articles/CBMi2gFBV...,ABT,NaN,ABT stock news,124,17,...,0.666667,16,3,0,1,15.372176,0.910640,0.0000,NaN,NaN


## 3. Prepare Features and Target

In [4]:
# Define target variable (sentiment direction: positive/neutral/negative)
def create_sentiment_target(sentiment_score, threshold=0.1):
    """Convert continuous sentiment to categorical target"""
    if sentiment_score > threshold:
        return 'positive'
    elif sentiment_score < -threshold:
        return 'negative'
    else:
        return 'neutral'

df['sentiment_direction'] = df['finbert_sentiment'].apply(create_sentiment_target)

print(" Target distribution:")
print(df['sentiment_direction'].value_counts())
print(f"\n{df['sentiment_direction'].value_counts(normalize=True) * 100}")

# Select features for modeling
feature_cols = [
    # Sentiment features
    'sentiment_momentum', 'sentiment_ma_7d', 'sentiment_volatility',
    'sentiment_extremity', 'sentiment_consensus',
    
    # Entity features
    'entity_density', 'entity_diversity', 'org_prominence', 'entity_freq_7d',
    
    # Event features
    'event_impact_score', 'has_high_impact_event', 'event_freq_7d',
    'event_sentiment_alignment',
    
    # Topic features
    'topic_entropy', 'topic_dominance',
    
    # Temporal features
    'is_market_hours', 'recency_score',
    
    # News velocity
    'article_count_7d', 'news_acceleration', 'news_burst',
    
    # Text complexity
    'readability_score', 'word_count',
    
    # Composite features
    'news_importance', 'attention_score', 'signal_strength'
]

# Filter to features that exist
feature_cols = [col for col in feature_cols if col in df.columns]

print(f"\n Selected {len(feature_cols)} features for modeling")
print(f"Features: {feature_cols[:10]}...")

 Target distribution:
sentiment_direction
positive    28
neutral     21
negative    14
Name: count, dtype: int64

sentiment_direction
positive    44.444444
neutral     33.333333
negative    22.222222
Name: proportion, dtype: float64

 Selected 18 features for modeling
Features: ['sentiment_momentum', 'sentiment_ma_7d', 'sentiment_volatility', 'sentiment_extremity', 'sentiment_consensus', 'entity_density', 'entity_diversity', 'org_prominence', 'entity_freq_7d', 'event_impact_score']...


## 4. Train-Test Split (Time-Aware)

In [ ]:
# Time-aware split (no data leakage)
df = df.sort_values('date').reset_index(drop=True)

# Use 80% for training, 20% for testing
split_idx = int(len(df) * 0.8)

train_df = df.iloc[:split_idx]
test_df = df.iloc[split_idx:]

# Prepare X and y with label encoding for compatibility
from sklearn.preprocessing import LabelEncoder

label_encoder = LabelEncoder()
X_train = train_df[feature_cols].values
y_train_raw = train_df['sentiment_direction'].values
y_train = label_encoder.fit_transform(y_train_raw)

X_test = test_df[feature_cols].values
y_test_raw = test_df['sentiment_direction'].values
y_test = label_encoder.transform(y_test_raw)

# Save label encoder for later use
import joblib
joblib.dump(label_encoder, MODELS_DIR / 'label_encoder.pkl')

print(f" Data split (time-aware):")
print(f"  Training set: {len(train_df):,} samples ({len(train_df)/len(df)*100:.1f}%)")
print(f"    Date range: {train_df['date'].min()} to {train_df['date'].max()}")
print(f"  Test set: {len(test_df):,} samples ({len(test_df)/len(df)*100:.1f}%)")
print(f"    Date range: {test_df['date'].min()} to {test_df['date'].max()}")

print(f"\n Feature matrix shapes:")
print(f"  X_train: {X_train.shape}")
print(f"  X_test: {X_test.shape}")
print(f"  y_train: {y_train.shape}")
print(f"  y_test: {y_test.shape}")

print(f"\nLabel encoding:")
print(f"  Classes: {label_encoder.classes_}")
print(f"  Encoded: {list(range(len(label_encoder.classes_)))}")
print(f"  Train distribution: {np.bincount(y_train)}")
print(f"  Test distribution: {np.bincount(y_test)}")

  Data split (time-aware):
  Training set: 50 samples (79.4%)
    Date range: 2025-09-26 07:00:00 to 2026-01-16 22:00:00
  Test set: 13 samples (20.6%)
    Date range: 2026-01-17 09:01:41 to 2026-01-18 12:19:48

  Feature matrix shapes:
  X_train: (50, 18)
  X_test: (13, 18)
  y_train: (50,)
  y_test: (13,)

 ️ Label encoding:
  Classes: ['negative' 'neutral' 'positive']
  Encoded: [0, 1, 2]
  Train distribution: [12 15 23]
  Test distribution: [2 6 5]


## 5. Feature Scaling

In [13]:
# Standardize features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Save scaler
joblib.dump(scaler, MODELS_DIR / 'feature_scaler.pkl')

print(" Features standardized")
print(f" Scaler saved to {MODELS_DIR / 'feature_scaler.pkl'}")
print(f"\n Scaled feature statistics:")
print(f"  Mean: {X_train_scaled.mean(axis=0)[:5]}...")
print(f"  Std: {X_train_scaled.std(axis=0)[:5]}...")

 Features standardized
 Scaler saved to c:\Users\HARSHIT\Desktop\p\nlp analysis\04_MODELS\feature_scaler.pkl

 Scaled feature statistics:
  Mean: [           nan 3.66373598e-17            nan 0.00000000e+00
 0.00000000e+00]...
  Std: [nan  1. nan  1.  1.]...


In [14]:
# Check for NaN values before training
print("\n Checking for NaN values...")
print(f"NaN in X_train_scaled: {np.isnan(X_train_scaled).sum()}")
print(f"NaN in X_test_scaled: {np.isnan(X_test_scaled).sum()}")

# If NaN values found, impute them with column mean
if np.isnan(X_train_scaled).sum() > 0 or np.isnan(X_test_scaled).sum() > 0:
    print("\n Found NaN values in feature data. Applying mean imputation...")
    from sklearn.impute import SimpleImputer
    
    imputer = SimpleImputer(strategy='mean')
    X_train_scaled = imputer.fit_transform(X_train_scaled)
    X_test_scaled = imputer.transform(X_test_scaled)
    
    print(f" After imputation - NaN in X_train_scaled: {np.isnan(X_train_scaled).sum()}")
    print(f"After imputation - NaN in X_test_scaled: {np.isnan(X_test_scaled).sum()}")
else:
    print(" No NaN values found. Data is clean!")


 Checking for NaN values...
NaN in X_train_scaled: 36
NaN in X_test_scaled: 2

 Found NaN values in feature data. Applying mean imputation...
 After imputation - NaN in X_train_scaled: 0
After imputation - NaN in X_test_scaled: 0


## 6. Train Base Models

In [15]:
print(" Training base models...\n")

# Initialize models with config parameters
base_models = {
    'Logistic Regression': LogisticRegression(
        C=1.0, max_iter=1000, class_weight='balanced', random_state=42
    ),
    'Random Forest': RandomForestClassifier(
        n_estimators=200, max_depth=20, min_samples_split=10,
        class_weight='balanced', random_state=42, n_jobs=-1
    ),
    'Gradient Boosting': GradientBoostingClassifier(
        n_estimators=200, learning_rate=0.1, max_depth=5,
        subsample=0.8, random_state=42
    ),
    'XGBoost': XGBClassifier(
        n_estimators=200, learning_rate=0.1, max_depth=7,
        subsample=0.8, colsample_bytree=0.8,
        random_state=42, eval_metric='logloss', use_label_encoder=False
    ),
    'LightGBM': LGBMClassifier(
        n_estimators=200, learning_rate=0.1, max_depth=7,
        subsample=0.8, colsample_bytree=0.8,
        random_state=42, verbose=-1
    ),
    'SVM': SVC(
        C=1.0, kernel='rbf', class_weight='balanced',
        probability=True, random_state=42
    ),
    'Naive Bayes': GaussianNB(),
    'KNN': KNeighborsClassifier(
        n_neighbors=7, weights='distance', n_jobs=-1
    )
}

# Train and evaluate each model
results = {}

for name, model in tqdm(base_models.items(), desc="Training models"):
    print(f"\n{'='*70}")
    print(f"Training: {name}")
    print(f"{'='*70}")
    
    # Train
    model.fit(X_train_scaled, y_train)
    
    # Predictions
    y_pred_train = model.predict(X_train_scaled)
    y_pred_test = model.predict(X_test_scaled)
    
    # Probabilities (for some models)
    try:
        y_prob_test = model.predict_proba(X_test_scaled)
    except:
        y_prob_test = None
    
    # Evaluate
    train_acc = accuracy_score(y_train, y_pred_train)
    test_acc = accuracy_score(y_test, y_pred_test)
    precision = precision_score(y_test, y_pred_test, average='weighted', zero_division=0)
    recall = recall_score(y_test, y_pred_test, average='weighted', zero_division=0)
    f1 = f1_score(y_test, y_pred_test, average='weighted', zero_division=0)
    
    results[name] = {
        'model': model,
        'train_accuracy': train_acc,
        'test_accuracy': test_acc,
        'precision': precision,
        'recall': recall,
        'f1_score': f1,
        'predictions': y_pred_test,
        'probabilities': y_prob_test
    }
    
    print(f"  Train Accuracy: {train_acc:.4f}")
    print(f"  Test Accuracy:  {test_acc:.4f}")
    print(f"  Precision:      {precision:.4f}")
    print(f"  Recall:         {recall:.4f}")
    print(f"  F1-Score:       {f1:.4f}")
    
    # Save model
    model_filename = name.lower().replace(' ', '_') + '.pkl'
    joblib.dump(model, BASE_MODELS_DIR / model_filename)
    print(f"   Saved to {BASE_MODELS_DIR / model_filename}")


 Training base models...



Training models:   0%|          | 0/8 [00:00<?, ?it/s]


Training: Logistic Regression
  Train Accuracy: 1.0000
  Test Accuracy:  0.8462
  Precision:      0.8590
  Recall:         0.8462
  F1-Score:       0.8462
   Saved to c:\Users\HARSHIT\Desktop\p\nlp analysis\04_MODELS\base_models\logistic_regression.pkl

Training: Random Forest
  Train Accuracy: 1.0000
  Test Accuracy:  0.7692
  Precision:      0.7179
  Recall:         0.7692
  F1-Score:       0.7413
   Saved to c:\Users\HARSHIT\Desktop\p\nlp analysis\04_MODELS\base_models\random_forest.pkl

Training: Gradient Boosting
  Train Accuracy: 1.0000
  Test Accuracy:  0.8462
  Precision:      0.8462
  Recall:         0.8462
  F1-Score:       0.8462
   Saved to c:\Users\HARSHIT\Desktop\p\nlp analysis\04_MODELS\base_models\gradient_boosting.pkl

Training: XGBoost
  Train Accuracy: 1.0000
  Test Accuracy:  0.9231
  Precision:      0.9359
  Recall:         0.9231
  F1-Score:       0.9138
   Saved to c:\Users\HARSHIT\Desktop\p\nlp analysis\04_MODELS\base_models\xgboost.pkl

Training: LightGBM
  Tr

## 7. Compare Base Model Performance

In [16]:
# Create performance comparison DataFrame
performance_df = pd.DataFrame([
    {
        'Model': name,
        'Train Accuracy': results[name]['train_accuracy'],
        'Test Accuracy': results[name]['test_accuracy'],
        'Precision': results[name]['precision'],
        'Recall': results[name]['recall'],
        'F1-Score': results[name]['f1_score'],
        'Overfit': results[name]['train_accuracy'] - results[name]['test_accuracy']
    }
    for name in results.keys()
]).sort_values('Test Accuracy', ascending=False)

print(" Base Model Performance Comparison:")
print("=" * 100)
print(performance_df.to_string(index=False))

# Save performance comparison
performance_df.to_csv(OUTPUTS_DIR / 'base_model_performance.csv', index=False)
print(f"\n Saved to {OUTPUTS_DIR / 'base_model_performance.csv'}")

# Visualize
fig = go.Figure()

fig.add_trace(go.Bar(
    x=performance_df['Model'],
    y=performance_df['Test Accuracy'],
    name='Test Accuracy',
    marker_color='lightblue'
))

fig.add_trace(go.Bar(
    x=performance_df['Model'],
    y=performance_df['F1-Score'],
    name='F1-Score',
    marker_color='coral'
))

fig.update_layout(
    title='Base Model Performance Comparison',
    xaxis_title='Model',
    yaxis_title='Score',
    barmode='group',
    height=500
)

fig.write_html(VIZ_DIR / 'base_model_comparison.html')
fig.show()

 Base Model Performance Comparison:
              Model  Train Accuracy  Test Accuracy  Precision   Recall  F1-Score  Overfit
            XGBoost            1.00       0.923077   0.935897 0.923077  0.913753 0.076923
Logistic Regression            1.00       0.846154   0.858974 0.846154  0.846154 0.153846
  Gradient Boosting            1.00       0.846154   0.846154 0.846154  0.846154 0.153846
           LightGBM            1.00       0.846154   0.846154 0.846154  0.846154 0.153846
                SVM            0.98       0.846154   0.890110 0.846154  0.842657 0.133846
                KNN            1.00       0.846154   0.890110 0.846154  0.842657 0.153846
      Random Forest            1.00       0.769231   0.717949 0.769231  0.741259 0.230769
        Naive Bayes            0.56       0.538462   0.500000 0.538462  0.481119 0.021538

 Saved to c:\Users\HARSHIT\Desktop\p\nlp analysis\03_RESULTS\outputs\base_model_performance.csv


## 8. Ensemble Method 1: Hard Voting

In [17]:
print(" Training Hard Voting Ensemble...\n")

# Select top 3 models for voting
voting_models = [
    ('lr', results['Logistic Regression']['model']),
    ('rf', results['Random Forest']['model']),
    ('xgb', results['XGBoost']['model'])
]

voting_hard = VotingClassifier(
    estimators=voting_models,
    voting='hard'
)

voting_hard.fit(X_train_scaled, y_train)
y_pred_voting_hard = voting_hard.predict(X_test_scaled)

acc_voting_hard = accuracy_score(y_test, y_pred_voting_hard)
f1_voting_hard = f1_score(y_test, y_pred_voting_hard, average='weighted')

print(f" Hard Voting Ensemble:")
print(f"  Test Accuracy: {acc_voting_hard:.4f}")
print(f"  F1-Score: {f1_voting_hard:.4f}")

# Save
joblib.dump(voting_hard, ENSEMBLE_DIR / 'voting_hard.pkl')
print(f" Saved to {ENSEMBLE_DIR / 'voting_hard.pkl'}")

 Training Hard Voting Ensemble...

 Hard Voting Ensemble:
  Test Accuracy: 0.9231
  F1-Score: 0.9138
 Saved to c:\Users\HARSHIT\Desktop\p\nlp analysis\04_MODELS\ensemble\voting_hard.pkl


## 9. Ensemble Method 2: Soft Voting (Weighted)

In [18]:
print(" Training Soft Voting Ensemble (Weighted)...\n")

# Use all models with probability support
voting_models_all = [
    ('lr', results['Logistic Regression']['model']),
    ('rf', results['Random Forest']['model']),
    ('gb', results['Gradient Boosting']['model']),
    ('xgb', results['XGBoost']['model']),
    ('lgb', results['LightGBM']['model']),
    ('svm', results['SVM']['model']),
    ('nb', results['Naive Bayes']['model']),
    ('knn', results['KNN']['model'])
]

# Weights based on test accuracy
weights = [
    results['Logistic Regression']['test_accuracy'],
    results['Random Forest']['test_accuracy'],
    results['Gradient Boosting']['test_accuracy'],
    results['XGBoost']['test_accuracy'],
    results['LightGBM']['test_accuracy'],
    results['SVM']['test_accuracy'],
    results['Naive Bayes']['test_accuracy'],
    results['KNN']['test_accuracy']
]

voting_soft = VotingClassifier(
    estimators=voting_models_all,
    voting='soft',
    weights=weights
)

voting_soft.fit(X_train_scaled, y_train)
y_pred_voting_soft = voting_soft.predict(X_test_scaled)

acc_voting_soft = accuracy_score(y_test, y_pred_voting_soft)
f1_voting_soft = f1_score(y_test, y_pred_voting_soft, average='weighted')

print(f" Soft Voting Ensemble (Weighted):")
print(f"  Test Accuracy: {acc_voting_soft:.4f}")
print(f"  F1-Score: {f1_voting_soft:.4f}")
print(f"  Weights: {[f'{w:.3f}' for w in weights]}")

# Save
joblib.dump(voting_soft, ENSEMBLE_DIR / 'voting_soft.pkl')
print(f" Saved to {ENSEMBLE_DIR / 'voting_soft.pkl'}")

 Training Soft Voting Ensemble (Weighted)...

 Soft Voting Ensemble (Weighted):
  Test Accuracy: 0.9231
  F1-Score: 0.9138
  Weights: ['0.846', '0.769', '0.846', '0.923', '0.846', '0.846', '0.538', '0.846']
 Saved to c:\Users\HARSHIT\Desktop\p\nlp analysis\04_MODELS\ensemble\voting_soft.pkl


## 10. Ensemble Method 3: Bagging

In [19]:
print(" Training Bagging Ensemble...\n")

bagging = BaggingClassifier(
    estimator=results['Random Forest']['model'],
    n_estimators=10,
    max_samples=0.8,
    max_features=0.8,
    bootstrap=True,
    random_state=42,
    n_jobs=-1
)

bagging.fit(X_train_scaled, y_train)
y_pred_bagging = bagging.predict(X_test_scaled)

acc_bagging = accuracy_score(y_test, y_pred_bagging)
f1_bagging = f1_score(y_test, y_pred_bagging, average='weighted')

print(f" Bagging Ensemble:")
print(f"  Test Accuracy: {acc_bagging:.4f}")
print(f"  F1-Score: {f1_bagging:.4f}")

# Save
joblib.dump(bagging, ENSEMBLE_DIR / 'bagging.pkl')
print(f" Saved to {ENSEMBLE_DIR / 'bagging.pkl'}")

 Training Bagging Ensemble...

 Bagging Ensemble:
  Test Accuracy: 0.7692
  F1-Score: 0.7413
 Saved to c:\Users\HARSHIT\Desktop\p\nlp analysis\04_MODELS\ensemble\bagging.pkl


## 11. Ensemble Method 4: Stacking (Meta-Learner)

In [20]:
print(" Training Stacking Ensemble (Meta-Learner)...\n")

# Use top 5 models as base estimators
stacking_models = [
    ('lr', results['Logistic Regression']['model']),
    ('rf', results['Random Forest']['model']),
    ('gb', results['Gradient Boosting']['model']),
    ('xgb', results['XGBoost']['model']),
    ('lgb', results['LightGBM']['model'])
]

# Meta-learner: Logistic Regression
meta_learner = LogisticRegression(C=1.0, random_state=42)

stacking = StackingClassifier(
    estimators=stacking_models,
    final_estimator=meta_learner,
    cv=5,
    stack_method='predict_proba'
)

stacking.fit(X_train_scaled, y_train)
y_pred_stacking = stacking.predict(X_test_scaled)

acc_stacking = accuracy_score(y_test, y_pred_stacking)
f1_stacking = f1_score(y_test, y_pred_stacking, average='weighted')
precision_stacking = precision_score(y_test, y_pred_stacking, average='weighted')
recall_stacking = recall_score(y_test, y_pred_stacking, average='weighted')

print(f" Stacking Ensemble (Meta-Learner):")
print(f"  Test Accuracy: {acc_stacking:.4f}")
print(f"  Precision: {precision_stacking:.4f}")
print(f"  Recall: {recall_stacking:.4f}")
print(f"  F1-Score: {f1_stacking:.4f}")

# Save
joblib.dump(stacking, ENSEMBLE_DIR / 'stacking.pkl')
print(f" Saved to {ENSEMBLE_DIR / 'stacking.pkl'}")

 Training Stacking Ensemble (Meta-Learner)...

 Stacking Ensemble (Meta-Learner):
  Test Accuracy: 0.9231
  Precision: 0.9359
  Recall: 0.9231
  F1-Score: 0.9138
 Saved to c:\Users\HARSHIT\Desktop\p\nlp analysis\04_MODELS\ensemble\stacking.pkl


## 12. Compare All Models

In [21]:
# Add ensemble results to comparison
ensemble_results = pd.DataFrame([
    {
        'Model': 'Hard Voting',
        'Type': 'Ensemble',
        'Test Accuracy': acc_voting_hard,
        'F1-Score': f1_voting_hard
    },
    {
        'Model': 'Soft Voting',
        'Type': 'Ensemble',
        'Test Accuracy': acc_voting_soft,
        'F1-Score': f1_voting_soft
    },
    {
        'Model': 'Bagging',
        'Type': 'Ensemble',
        'Test Accuracy': acc_bagging,
        'F1-Score': f1_bagging
    },
    {
        'Model': 'Stacking',
        'Type': 'Ensemble',
        'Test Accuracy': acc_stacking,
        'F1-Score': f1_stacking
    }
])

# Combine with base models
base_results = performance_df[['Model', 'Test Accuracy', 'F1-Score']].copy()
base_results['Type'] = 'Base Model'

all_results = pd.concat([base_results, ensemble_results], ignore_index=True)
all_results = all_results.sort_values('Test Accuracy', ascending=False)

print(" Complete Model Performance Comparison:")
print("=" * 80)
print(all_results.to_string(index=False))

# Save
all_results.to_csv(OUTPUTS_DIR / 'all_model_performance.csv', index=False)
print(f"\n Saved to {OUTPUTS_DIR / 'all_model_performance.csv'}")

# Visualize
fig = px.bar(
    all_results,
    x='Model',
    y='Test Accuracy',
    color='Type',
    title='Complete Model Performance: Base Models vs Ensembles',
    labels={'Test Accuracy': 'Test Accuracy', 'Model': 'Model Name'},
    text='Test Accuracy'
)

fig.update_traces(texttemplate='%{text:.3f}', textposition='outside')
fig.update_layout(height=600, xaxis_tickangle=-45)

fig.write_html(VIZ_DIR / 'all_models_comparison.html')
fig.show()

# Find best model
best_model = all_results.iloc[0]
print(f"\n Best Model: {best_model['Model']}")
print(f"  Type: {best_model['Type']}")
print(f"  Test Accuracy: {best_model['Test Accuracy']:.4f}")
print(f"  F1-Score: {best_model['F1-Score']:.4f}")

if best_model['Test Accuracy'] >= 0.889:
    print(f"   EXCELLENT - Exceeds target (88.9%)")
elif best_model['Test Accuracy'] >= 0.85:
    print(f"   GOOD - Meets industry benchmark (85-90%)")
else:
    print(f"   NEEDS IMPROVEMENT")

 Complete Model Performance Comparison:
              Model  Test Accuracy  F1-Score       Type
            XGBoost       0.923077  0.913753 Base Model
        Hard Voting       0.923077  0.913753   Ensemble
        Soft Voting       0.923077  0.913753   Ensemble
           Stacking       0.923077  0.913753   Ensemble
Logistic Regression       0.846154  0.846154 Base Model
  Gradient Boosting       0.846154  0.846154 Base Model
           LightGBM       0.846154  0.846154 Base Model
                SVM       0.846154  0.842657 Base Model
                KNN       0.846154  0.842657 Base Model
      Random Forest       0.769231  0.741259 Base Model
            Bagging       0.769231  0.741259   Ensemble
        Naive Bayes       0.538462  0.481119 Base Model

 Saved to c:\Users\HARSHIT\Desktop\p\nlp analysis\03_RESULTS\outputs\all_model_performance.csv



 Best Model: XGBoost
  Type: Base Model
  Test Accuracy: 0.9231
  F1-Score: 0.9138
   EXCELLENT - Exceeds target (88.9%)


## 13. Confusion Matrix (Best Model)

In [22]:
# Confusion matrix for stacking (typically best)
cm = confusion_matrix(y_test, y_pred_stacking)
labels = sorted(set(y_test))

# Visualize
fig = px.imshow(
    cm,
    labels=dict(x="Predicted", y="Actual", color="Count"),
    x=labels,
    y=labels,
    text_auto=True,
    color_continuous_scale='Blues',
    title='Confusion Matrix - Stacking Ensemble'
)

fig.update_layout(height=600)
fig.write_html(VIZ_DIR / 'confusion_matrix_stacking.html')
fig.show()

# Classification report
print("\n Classification Report (Stacking Ensemble):")
print("=" * 70)
print(classification_report(y_test, y_pred_stacking))


 Classification Report (Stacking Ensemble):
              precision    recall  f1-score   support

           0       1.00      0.50      0.67         2
           1       1.00      1.00      1.00         6
           2       0.83      1.00      0.91         5

    accuracy                           0.92        13
   macro avg       0.94      0.83      0.86        13
weighted avg       0.94      0.92      0.91        13



## 14. Feature Importance (Best Tree-Based Model)

In [23]:
# Get feature importance from best tree-based model
best_tree_model = results['XGBoost']['model']

if hasattr(best_tree_model, 'feature_importances_'):
    importances = best_tree_model.feature_importances_
    
    feature_importance_df = pd.DataFrame({
        'feature': feature_cols,
        'importance': importances
    }).sort_values('importance', ascending=False)
    
    print(" Top 15 Most Important Features (XGBoost):")
    print("=" * 70)
    print(feature_importance_df.head(15).to_string(index=False))
    
    # Save
    feature_importance_df.to_csv(OUTPUTS_DIR / 'feature_importance.csv', index=False)
    
    # Visualize
    fig = go.Figure(data=[
        go.Bar(
            y=feature_importance_df.head(20)['feature'][::-1],
            x=feature_importance_df.head(20)['importance'][::-1],
            orientation='h',
            marker=dict(color=feature_importance_df.head(20)['importance'][::-1], colorscale='Viridis')
        )
    ])
    
    fig.update_layout(
        title='Top 20 Feature Importances (XGBoost)',
        xaxis_title='Importance',
        yaxis_title='Feature',
        height=700
    )
    
    fig.write_html(VIZ_DIR / 'feature_importance.html')
    fig.show()
    
    print(f"\n Saved to {OUTPUTS_DIR / 'feature_importance.csv'}")

 Top 15 Most Important Features (XGBoost):
             feature  importance
 sentiment_consensus    0.222311
 sentiment_extremity    0.211774
     sentiment_ma_7d    0.130093
  sentiment_momentum    0.082330
    entity_diversity    0.072146
       event_freq_7d    0.058823
          word_count    0.036994
sentiment_volatility    0.033450
     is_market_hours    0.031450
       topic_entropy    0.028539
     topic_dominance    0.025354
      org_prominence    0.024681
       recency_score    0.021183
      entity_freq_7d    0.012435
      entity_density    0.008435



 Saved to c:\Users\HARSHIT\Desktop\p\nlp analysis\03_RESULTS\outputs\feature_importance.csv


## 15. Summary Report

In [24]:
# Create comprehensive summary
summary = {
    'dataset': {
        'total_samples': len(df),
        'train_samples': len(train_df),
        'test_samples': len(test_df),
        'num_features': len(feature_cols)
    },
    'base_models': {
        'count': len(base_models),
        'models': list(base_models.keys()),
        'best_base_model': performance_df.iloc[0]['Model'],
        'best_base_accuracy': float(performance_df.iloc[0]['Test Accuracy'])
    },
    'ensemble_methods': {
        'count': 4,
        'methods': ['Hard Voting', 'Soft Voting', 'Bagging', 'Stacking'],
        'hard_voting_accuracy': float(acc_voting_hard),
        'soft_voting_accuracy': float(acc_voting_soft),
        'bagging_accuracy': float(acc_bagging),
        'stacking_accuracy': float(acc_stacking)
    },
    'best_overall': {
        'model': best_model['Model'],
        'type': best_model['Type'],
        'accuracy': float(best_model['Test Accuracy']),
        'f1_score': float(best_model['F1-Score'])
    },
    'benchmark_comparison': {
        'industry_benchmark': '85-90%',
        'target': '88.9%',
        'achieved': float(best_model['Test Accuracy']),
        'meets_target': bool(best_model['Test Accuracy'] >= 0.889)
    }
}

# Save summary
with open(OUTPUTS_DIR / 'ensemble_modeling_summary.json', 'w') as f:
    json.dump(summary, f, indent=2, default=str)

print(" Ensemble Modeling Summary")
print("=" * 70)
print(f"Dataset: {summary['dataset']['total_samples']:,} samples")
print(f"Features: {summary['dataset']['num_features']}")
print(f"\nBase Models: {summary['base_models']['count']}")
print(f"  Best: {summary['base_models']['best_base_model']} ({summary['base_models']['best_base_accuracy']:.4f})")
print(f"\nEnsemble Methods: {summary['ensemble_methods']['count']}")
print(f"  Hard Voting: {summary['ensemble_methods']['hard_voting_accuracy']:.4f}")
print(f"  Soft Voting: {summary['ensemble_methods']['soft_voting_accuracy']:.4f}")
print(f"  Bagging: {summary['ensemble_methods']['bagging_accuracy']:.4f}")
print(f"  Stacking: {summary['ensemble_methods']['stacking_accuracy']:.4f}")
print(f"\n Best Overall: {summary['best_overall']['model']}")
print(f"  Accuracy: {summary['best_overall']['accuracy']:.4f}")
print(f"  F1-Score: {summary['best_overall']['f1_score']:.4f}")
print(f"  Meets Target: {' YES' if summary['benchmark_comparison']['meets_target'] else ' NO'}")

print(f"\n Summary saved to {OUTPUTS_DIR / 'ensemble_modeling_summary.json'}")
print("\n Ensemble Modeling complete!")
print("\n Output files:")
print(f"  - Models: {BASE_MODELS_DIR}/*.pkl (8 base models)")
print(f"  - Ensembles: {ENSEMBLE_DIR}/*.pkl (4 ensemble methods)")
print(f"  - {OUTPUTS_DIR / 'all_model_performance.csv'}")
print(f"  - {OUTPUTS_DIR / 'feature_importance.csv'}")
print(f"  - {VIZ_DIR / 'all_models_comparison.html'}")
print(f"  - {VIZ_DIR / 'confusion_matrix_stacking.html'}")
print(f"  - {VIZ_DIR / 'feature_importance.html'}")

 Ensemble Modeling Summary
Dataset: 63 samples
Features: 18

Base Models: 8
  Best: XGBoost (0.9231)

Ensemble Methods: 4
  Hard Voting: 0.9231
  Soft Voting: 0.9231
  Bagging: 0.7692
  Stacking: 0.9231

 Best Overall: XGBoost
  Accuracy: 0.9231
  F1-Score: 0.9138
  Meets Target:  YES

 Summary saved to c:\Users\HARSHIT\Desktop\p\nlp analysis\03_RESULTS\outputs\ensemble_modeling_summary.json

 Ensemble Modeling complete!

 Output files:
  - Models: c:\Users\HARSHIT\Desktop\p\nlp analysis\04_MODELS\base_models/*.pkl (8 base models)
  - Ensembles: c:\Users\HARSHIT\Desktop\p\nlp analysis\04_MODELS\ensemble/*.pkl (4 ensemble methods)
  - c:\Users\HARSHIT\Desktop\p\nlp analysis\03_RESULTS\outputs\all_model_performance.csv
  - c:\Users\HARSHIT\Desktop\p\nlp analysis\03_RESULTS\outputs\feature_importance.csv
  - c:\Users\HARSHIT\Desktop\p\nlp analysis\03_RESULTS\visualizations\all_models_comparison.html
  - c:\Users\HARSHIT\Desktop\p\nlp analysis\03_RESULTS\visualizations\confusion_matrix_sta